DATA PREPARATION

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
data= 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'

In [3]:
df= pd.read_csv(data)
df.to_csv("course_lead_scoring_2026.csv", index=False)

In [4]:
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [5]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [6]:
df.dtypes

lead_source                  object
industry                     object
employment_status            object
location                     object
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [10]:
categorical=['lead_source','industry','employment_status','location']
numerical= ['annual_income','number_of_courses_viewed','interaction_count','lead_score']

In [11]:
df[categorical]= df[categorical].fillna('NA')
df[numerical]= df[numerical].fillna(0.0)

In [12]:
df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

In [13]:
# QUESTION 1
df['industry'].mode()

0    technology
Name: industry, dtype: object

In [15]:
# QUESTION 2
from sklearn.model_selection import train_test_split

In [16]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

In [17]:
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

In [18]:
del df_train['converted']
del df_val['converted']
del df_test['converted']

In [19]:
len(df_full_train),len(df_train),len(df_val),len(df_test)

(4000, 3000, 1000, 1000)

In [20]:
df_full_train[numerical].corrwith(df_full_train.converted)

annual_income               0.188437
number_of_courses_viewed    0.393200
interaction_count           0.450273
lead_score                  0.482078
dtype: float64

In [22]:
corr= df[numerical].corr()
corr

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


In [23]:
pairs = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count'),
]
for a, b in pairs:
    print(a, '&', b, '->', corr.loc[a, b])

interaction_count & lead_score -> 0.9157457980418697
number_of_courses_viewed & lead_score -> 0.7572042392104886
number_of_courses_viewed & interaction_count -> 0.7216090038937035
annual_income & interaction_count -> 0.12284235135955959


In [24]:
# QUESTION 3

from sklearn.metrics import mutual_info_score

In [25]:
mutual_info_score(df_full_train.converted, df_full_train.lead_source)

0.0347609795100606

In [26]:
def mutual_info_converted_score(series):
    return mutual_info_score(series, df_full_train.converted)

In [27]:
mi = df_full_train[categorical].apply(mutual_info_converted_score)
mi.round(2).sort_values(ascending=False)

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

In [28]:
# QUESTION 4
dicts = df[categorical + numerical].to_dict(orient='records')

In [29]:
from sklearn.feature_extraction import DictVectorizer

dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

In [30]:
X_train.shape

(3000, 28)

In [31]:
val_dict = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dict)

In [32]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')

In [35]:
y_pred = model.predict(X_val)

accuracy = float((y_val == y_pred).mean())
round(accuracy,2)

0.65

In [38]:
# QUESTION 5 
features = categorical + numerical

def train_and_score(feature_list):
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(df_train[feature_list].to_dict(orient='records'))
    X_val = dv.transform(df_val[feature_list].to_dict(orient='records'))

    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)
    return float((y_val == y_pred).mean())


In [39]:
# Baseline with all features (unrounded)
base_accuracy = train_and_score(features)
print('baseline:', base_accuracy)

# Drop each candidate feature and compare
for f in ['lead_source', 'number_of_courses_viewed', 'interaction_count']:
    reduced = [x for x in features if x != f]
    accuracy = train_and_score(reduced)
    print(f, 'accuracy =', accuracy, 'difference =', base_accuracy - accuracy)

baseline: 0.645
lead_source accuracy = 0.642 difference = 0.0030000000000000027
number_of_courses_viewed accuracy = 0.643 difference = 0.0020000000000000018
interaction_count accuracy = 0.601 difference = 0.04400000000000004


In [40]:
# QUESTION 6
dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(df_train[features].to_dict(orient='records'))
X_val = dv.transform(df_val[features].to_dict(orient='records'))

results = {}
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)
    acc = float((y_val == y_pred).mean())
    results[C] = round(acc, 3)
    print(C, results[C])

1e-06 0.598
1e-05 0.598
0.0001 0.613
0.001 0.645


In [41]:
best_accuracy = max(results.values())
best_C = min(C for C, a in results.items() if a == best_accuracy)
print('Best C:', best_C)

Best C: 0.001
